In [29]:
import os
print(os.listdir("/kaggle/input/datasets/clmentbisaillon/fake-and-real-news-dataset"))

['True.csv', 'Fake.csv']


In [30]:
import pandas as pd

fake_df = pd.read_csv("/kaggle/input/datasets/clmentbisaillon/fake-and-real-news-dataset/Fake.csv")
true_df = pd.read_csv("/kaggle/input/datasets/clmentbisaillon/fake-and-real-news-dataset/True.csv")

print("Fake:", fake_df.shape)
print("True:", true_df.shape)
print(fake_df.head())
print(true_df["subject"].value_counts())
print(fake_df["subject"].value_counts())

Fake: (23481, 4)
True: (21417, 4)
                                               title  \
0   Donald Trump Sends Out Embarrassing New Year’...   
1   Drunk Bragging Trump Staffer Started Russian ...   
2   Sheriff David Clarke Becomes An Internet Joke...   
3   Trump Is So Obsessed He Even Has Obama’s Name...   
4   Pope Francis Just Called Out Donald Trump Dur...   

                                                text subject  \
0  Donald Trump just couldn t wish all Americans ...    News   
1  House Intelligence Committee Chairman Devin Nu...    News   
2  On Friday, it was revealed that former Milwauk...    News   
3  On Christmas day, Donald Trump announced that ...    News   
4  Pope Francis used his annual Christmas Day mes...    News   

                date  
0  December 31, 2017  
1  December 31, 2017  
2  December 30, 2017  
3  December 29, 2017  
4  December 25, 2017  
subject
politicsNews    11272
worldnews       10145
Name: count, dtype: int64
subject
News               9

In [31]:
!pip install transformers datasets torch scikit-learn -q

In [32]:
import re

# Vérifier l'artefact "(Reuters)" présent uniquement dans les vrais articles
print("True contenant '(Reuters)':", true_df["text"].str.contains(r"\(Reuters\)").mean())
print("Fake contenant '(Reuters)':", fake_df["text"].str.contains(r"\(Reuters\)").mean())

def clean_text(text):
    text = str(text)

    # Corrige l'artefact d'apostrophe manquante (ex: "Here s what" -> "Here's what")
    text = re.sub(r"(\w) s ", r"\1's ", text)
    text = re.sub(r"(\w) ve ", r"\1've ", text)
    text = re.sub(r"(\w) re ", r"\1're ", text)
    text = re.sub(r"(\w) ll ", r"\1'll ", text)
    text = re.sub(r"(\w) t ", r"\1't ", text)
    text = re.sub(r"(\w) d ", r"\1'd ", text)

    # Enlève "CITY (Reuters) - " en début de texte
    text = re.sub(r"^[A-Z\s,]+\(Reuters\)\s*-\s*", "", text)
    # Enlève les mentions type "Featured image via ..." souvent présentes dans Fake.csv
    text = re.sub(r"Featured image via.*", "", text, flags=re.IGNORECASE)
    # Enlève les URL et mentions Twitter
    text = re.sub(r"pic\.twitter\.com/\S+", "", text)
    text = re.sub(r"https?://\S+", "", text)

    # Normalise les guillemets/apostrophes typographiques
    text = text.replace("’", "'").replace("‘", "'").replace("“", '"').replace("”", '"')

    return text.strip()

# Vérification avant/après sur un échantillon
sample = fake_df["text"].iloc[0]
print("AVANT:", sample[:200])
print("APRÈS:", clean_text(sample)[:200])

fake_df["text"] = fake_df["text"].apply(clean_text)
true_df["text"] = true_df["text"].apply(clean_text)

print("✅ Nettoyage effectué")

True contenant '(Reuters)': 0.9920623803520567
Fake contenant '(Reuters)': 0.0003832886163280951
AVANT: Donald Trump just couldn t wish all Americans a Happy New Year and leave it at that. Instead, he had to give a shout out to his enemies, haters and  the very dishonest fake news media.  The former rea
APRÈS: Donald Trump just couldn't wish all Americans a Happy New Year and leave it at that. Instead, he had to give a shout out to his enemies, haters and  the very dishonest fake news media.  The former rea
✅ Nettoyage effectué


In [33]:
from sklearn.model_selection import train_test_split

def load_and_prepare(fake_df, true_df):
    fake_df = fake_df.copy()
    true_df = true_df.copy()

    fake_df["label"] = 1  # 1 = fake
    true_df["label"] = 0  # 0 = true

    df = pd.concat([fake_df, true_df], axis=0).reset_index(drop=True)

    df["content"] = df["title"].fillna("") + " " + df["text"].fillna("")
    df["content"] = df["content"].str.strip()
    df = df[df["content"].str.len() > 0]

    df = df.sample(frac=1, random_state=42).reset_index(drop=True)

    return df[["content", "label"]]

df = load_and_prepare(fake_df, true_df)

print(df.shape)
print(df["label"].value_counts())
print(df.head())

train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42, stratify=df["label"])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df["label"])

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

(44898, 2)
label
1    23481
0    21417
Name: count, dtype: int64
                                             content  label
0  Ben Stein Calls Out 9th Circuit Court: Committ...      1
1  Trump drops Steve Bannon from National Securit...      0
2  Puerto Rico expects U.S. to lift Jones Act shi...      0
3  OOPS: Trump Just Accidentally Confirmed He Lea...      1
4  Donald Trump heads for Scotland to reopen a go...      0
Train: (31428, 2)
Val: (6735, 2)
Test: (6735, 2)


In [34]:
import torch
from torch.utils.data import Dataset

class NewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
            "labels": torch.tensor(label, dtype=torch.long)
        }

In [35]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

train_dataset = NewsDataset(
    texts=train_df["content"].tolist(),
    labels=train_df["label"].tolist(),
    tokenizer=tokenizer
)

val_dataset = NewsDataset(
    texts=val_df["content"].tolist(),
    labels=val_df["label"].tolist(),
    tokenizer=tokenizer
)

test_dataset = NewsDataset(
    texts=test_df["content"].tolist(),
    labels=test_df["label"].tolist(),
    tokenizer=tokenizer
)

print("✅ Datasets créés")
print("Exemple:", train_dataset[0]["input_ids"].shape)

✅ Datasets créés
Exemple: torch.Size([256])


In [36]:
import torch
print("GPU disponible:", torch.cuda.is_available())
print("Nom du GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Aucun")

GPU disponible: True
Nom du GPU: Tesla T4


In [37]:
from transformers import BertConfig, BertForSequenceClassification

config = BertConfig.from_pretrained("bert-base-uncased", num_labels=2)
config.hidden_dropout_prob = 0.3          # défaut 0.1 -> 0.3
config.attention_probs_dropout_prob = 0.3  # défaut 0.1 -> 0.3

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    config=config
)

model = model.to("cuda" if torch.cuda.is_available() else "cpu")
print("✅ Modèle chargé avec dropout renforcé")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Modèle chargé avec dropout renforcé


In [38]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average="binary")
    acc = accuracy_score(labels, predictions)
    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

training_args = TrainingArguments(
    output_dir="/kaggle/working/results",
    num_train_epochs=2,                 # réduit de 3 à 2
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,                 # LR explicite plus bas
    warmup_ratio=0.1,
    weight_decay=0.05,                  # régularisation renforcée
    logging_dir="/kaggle/working/logs",
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("✅ Trainer configuré avec régularisation et early stopping")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


✅ Trainer configuré avec régularisation et early stopping


In [39]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.030944,0.193877,0.982925,1.000000,0.967348,0.983403
2,0.028787,0.142756,0.987676,1.000000,0.976434,0.988076


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=1966, training_loss=0.11427006325547111, metrics={'train_runtime': 2074.9129, 'train_samples_per_second': 30.293, 'train_steps_per_second': 0.948, 'total_flos': 8269054247854080.0, 'train_loss': 0.11427006325547111, 'epoch': 2.0})

Évaluer sur le test set

In [40]:
test_results = trainer.evaluate(test_dataset)
print(test_results)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 0.14427801966667175, 'eval_accuracy': 0.9879732739420936, 'eval_precision': 1.0, 'eval_recall': 0.9770082316207778, 'eval_f1': 0.9883704235463029, 'eval_runtime': 77.1048, 'eval_samples_per_second': 87.349, 'eval_steps_per_second': 1.375, 'epoch': 2.0}


Sauvegarder le modèle fine-tuné

In [41]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

vec = TfidfVectorizer(max_features=5000, stop_words="english")
X_train = vec.fit_transform(train_df["content"])
X_test = vec.transform(test_df["content"])

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, train_df["label"])

print("=== Baseline TF-IDF + Régression Logistique ===")
print(classification_report(test_df["label"], clf.predict(X_test)))
print("Si ce score est aussi proche de 99%, le dataset reste 'trop facile' malgré le nettoyage.")

=== Baseline TF-IDF + Régression Logistique ===
              precision    recall  f1-score   support

           0       0.98      0.98      0.98      3212
           1       0.99      0.98      0.98      3523

    accuracy                           0.98      6735
   macro avg       0.98      0.98      0.98      6735
weighted avg       0.98      0.98      0.98      6735

Si ce score est aussi proche de 99%, le dataset reste 'trop facile' malgré le nettoyage.


In [42]:
model.save_pretrained("/kaggle/working/bert_fake_news_model")
tokenizer.save_pretrained("/kaggle/working/bert_fake_news_model")
print("✅ Modèle sauvegardé")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Modèle sauvegardé


In [43]:
# Compare quelques échantillons bruts pour repérer d'autres patterns
print("=== FAKE ===")
for t in fake_df["text"].sample(5, random_state=1):
    print(t[:200], "\n---")

print("=== TRUE ===")
for t in true_df["text"].sample(5, random_state=1):
    print(t[:200], "\n---")

=== FAKE ===
Here's what Politico's headline today looked like:Here's what the leftist anti-American rag Politico had to say about Angela Merkel's visit to the US to meet with our new President Donald Trump: This  
---
Who is Paul Manafort? Well, for starters, he's a former principal lobbyist with the firm Black, Manafort, Stone and Kelly (BMS & K), a firm that had close ties to the Reagan and Bush White House and w 
---
Liberals  internal struggle in one clip: .@andersoncooper on the level, .@SenWarren name-calling. #Bannon   William Bairamian (@Bairamian) December 1, 2016 
---
It's no accident that President Obama named Vanita Gupta acting head of the Civil Rights Division of the DOJ. Gupta is beloved by the radical left for her militant hostility toward law enforcement off 
---
We've heard about Hillary Clinton's bad behavior before but this tell-all book goes right into details of what Clinton was like as First Lady  Hillary Clinton has a  Jekyll and Hyde  personality that  
---
==